# Klasifikasi Uang Koin Rupiah - EDA, Preprocessing, dan Custom CNN (5 Kelas)

**Cakupan Asistensi & Masukan Dosen:**
- **Penerapan Arsitektur Custom CNN:** Mengadopsi arsitektur dan metodologi Custom CNN dari `model/cnn/Rupiah_CustomCNN.ipynb` (uang kertas) untuk dataset koin Rupiah (`dataset_koin_baru`).
- **5 Kelas Target Koin:** `100`, `200`, `500`, `1000`, dan `sisi_garuda`.
- **Adaptasi Karakteristik Fisika Koin:**
  1. *Rotational Symmetry:* Koin dapat diputar dalam berbagai orientasi (0°–360°), sehingga augmentasi rotasi penuh sangat krusial.
  2. *Refleksi Spekular Logam & Keausan:* Menangani kilau reflektif permukaan logam koin dan variasi pencahayaan.
  3. *Audit & Anti-Kebocoran Data (Data Leakage Prevention):* Pengelompokan dHash 64-bit connected components dengan `StratifiedGroupKFold` 20-fold (70/15/15) agar frame rekaman dari koin yang sama tidak bocor antar train, validation, dan test set.
  4. *Evaluasi Komprehensif:* Confusion Matrix (absolut & recall ternormalisasi), Wilson 95% Confidence Interval untuk akurasi, Macro-F1, dan kurva riwayat pelatihan.
  5. *Logika Cerdas Sisi Balik & Dual-Sided Verification:* Knowledge mapping tokoh pahlawan nasional pada `sisi_garuda` untuk inferensi nominal dan verifikasi konsensus dua sisi.
  6. *Ekspor Deployment:* Konversi model terpilih ke format Keras (`.keras`) dan TensorFlow Lite (`.tflite` FP16) untuk siap diintegrasikan ke sistem serving/web app.

## 1. Setup dan Konfigurasi

Semua parameter penting dirangkum di satu tempat (`Config`) menggunakan `dataclass` agar terstruktur, mudah dikonfigurasi ulang, dan terdokumentasi lengkap dalam evaluasi akhir.

In [ ]:
import os, sys, io, json, math, random, shutil, hashlib, platform, subprocess, time, warnings
from pathlib import Path
from dataclasses import dataclass, asdict
from concurrent.futures import ThreadPoolExecutor

# Konfigurasi environment backend
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["MPLCONFIGDIR"] = os.path.join(os.environ.get("TEMP", "C:/tmp"), "mpl_cache")
os.makedirs(os.environ["MPLCONFIGDIR"], exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from IPython.display import display

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kw): return x

IN_COLAB = Path('/content').is_dir()
if IN_COLAB:
    BASE_DIR = Path('/content')
else:
    # Cari direktori kerja root frontend
    ws_candidate = Path(r"C:/Users/Bunga Alyssa/OneDrive/文档/DS/smt  7/deep l/uas/frontend")
    BASE_DIR = ws_candidate if ws_candidate.exists() else Path('.').resolve()


@dataclass(frozen=True)
class Config:
    # --- data
    seed: int = 42
    classes: tuple = ('100', '200', '500', '1000', 'sisi_garuda')
    img_size: tuple = (224, 224)
    split_ratios: tuple = (0.70, 0.15, 0.15)   # train, val, test
    n_group_folds: int = 20                    # 1 fold = 5% data; 14/3/3 fold -> 70/15/15
    near_dup_threshold: int = 6                # jarak Hamming dHash (0-64); makin kecil makin ketat
    resize_mode: str = 'pad'                   # 'pad' = pertahankan rasio + padding hitam | 'stretch'
    n_workers: int = min(8, os.cpu_count() or 2)
    # --- training
    batch_size: int = 32
    baseline_epochs: int = 15
    tuning_epochs: int = 30
    baseline_lr: float = 1e-3
    tuning_lr: float = 5e-4
    patience: int = 8
    # --- lokasi dataset
    data_dir_name: str = 'dataset_koin_baru'
    data_root_override: str = os.environ.get('COIN_DATA_ROOT', '')
    quick_test: bool = os.environ.get('COIN_QUICK', '') == '1'   # uji cepat bila ingin verifikasi pipeline


cfg = Config()
CLASSES = list(cfg.classes)
IMG_SIZE = tuple(cfg.img_size)
SEED = cfg.seed
random.seed(SEED)
np.random.seed(SEED)

OUT_ROOT = BASE_DIR / 'coin_processed'
REPORT_DIR = OUT_ROOT / 'eda_reports'
RESULTS = OUT_ROOT / 'training_results'
for d in (OUT_ROOT, REPORT_DIR, RESULTS):
    d.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .25,
                     'axes.spines.top': False, 'axes.spines.right': False})


def pretty(c):
    """'100' -> 'Rp 100', 'sisi_garuda' -> 'Sisi Garuda / Pahlawan'"""
    if c == 'sisi_garuda':
        return 'Sisi Garuda / Pahlawan'
    return 'Rp ' + f"{int(c):,}".replace(',', '.')


print('Konfigurasi Model & Data Koin:')
for k, v in asdict(cfg).items():
    print(f'  {k:20s} {v}')
if cfg.quick_test:
    print('\n>>> MODE UJI CEPAT AKTIF: epoch dipangkas, hasil akurasi TIDAK valid untuk laporan.')

## 2. Penemuan Dataset Koin

Dataset koin dimuat dari folder `dataset_koin_baru` yang memuat 5 subfolder kelas target: `100`, `200`, `500`, `1000`, dan `sisi_garuda`.

In [ ]:
if cfg.data_root_override:
    DATA_ROOT = Path(cfg.data_root_override)
else:
    candidates = [
        BASE_DIR / cfg.data_dir_name,
        BASE_DIR.parent / cfg.data_dir_name,
        Path(cfg.data_dir_name),
        Path('/content') / cfg.data_dir_name
    ]
    DATA_ROOT = next((d for d in candidates if d.exists() and all((d / c).is_dir() for c in CLASSES)), None)

if DATA_ROOT is None:
    raise FileNotFoundError(f'Folder 5 kelas koin tidak ditemukan di calon lokasi: {[str(c) for c in candidates]}')

print(f'DATA_ROOT terdeteksi : {DATA_ROOT.resolve()}')
print('Subfolder kelas terverifikasi:')
for c in CLASSES:
    n_files = len(list((DATA_ROOT / c).glob('*.*')))
    print(f'  - {c:12s} ({pretty(c):<25s}): {n_files} berkas')

## 3. Inventaris Dataset dan Audit Kualitas Citra (EDA)

Pada tahapan ini, setiap citra diperiksa integritasnya:
- **Validitas Berkas:** Mendeteksi citra rusak (*corrupted files*).
- **Dimensi & Rasio Aspek:** Resolusi asli dan orientasi citra.
- **Blur Laplacian:** Mengukur tingkat keburaman citra dengan *Variance of Laplacian* (kernel 4-tetangga). Makin kecil nilainya, citra makin buram.
- **Brightness & Contrast:** Rata-rata dan standar deviasi intensitas keabuan (grayscale).
- **Glare Ratio:** Proporsi piksel jenuh ($\ge 250$) untuk mendeteksi kilau reflektif permukaan logam koin.
- **dHash (Difference Hash 64-bit):** Sidik jari visual citra untuk audit near-duplicate.

In [ ]:
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}


def laplacian_variance(gray):
    """Variance of Laplacian (kernel 4-tetangga). Makin kecil -> makin blur."""
    g = gray.astype(np.float32)
    lap = -4 * g[1:-1, 1:-1] + g[:-2, 1:-1] + g[2:, 1:-1] + g[1:-1, :-2] + g[1:-1, 2:]
    return float(lap.var())


def dhash64(gray_img):
    """Difference hash 64-bit dari gambar grayscale PIL."""
    a = np.asarray(gray_img.resize((9, 8), Image.Resampling.LANCZOS), dtype=np.int16)
    bits = (a[:, 1:] > a[:, :-1]).flatten()
    h = 0
    for b in bits:
        h = (h << 1) | int(b)
    return h


def analyze_image(item):
    path, cls = item
    rec = {'path': str(path), 'filename': path.name, 'label': cls, 'valid': False, 'error': None}
    try:
        data = path.read_bytes()
        sha = hashlib.sha256(data).hexdigest()
        with Image.open(io.BytesIO(data)) as im:
            fmt, mode0 = im.format, im.mode
            orientation = im.getexif().get(0x0112, 1)
            w, h = im.size
            if orientation in (5, 6, 7, 8):          # EXIF rotasi 90 derajat
                w, h = h, w
            if w < 1 or h < 1:
                raise ValueError('Dimensi tidak valid')
            im.draft('RGB', (448, 448))
            rgb = ImageOps.exif_transpose(im).convert('RGB')
        gray = rgb.convert('L').resize(IMG_SIZE, Image.Resampling.BOX)
        g = np.asarray(gray, dtype=np.float32)
        rec.update(valid=True, width=w, height=h, mode=mode0, format=fmt,
                   aspect_ratio=w / h, file_kb=len(data) / 1024, sha256=sha,
                   brightness=float(g.mean()), contrast=float(g.std()),
                   blur_laplacian=laplacian_variance(g),
                   glare_ratio=float((g >= 250).mean()),
                   dhash=f'{dhash64(gray):016x}')
    except Exception as e:
        rec['error'] = f'{type(e).__name__}: {e}'
    return rec


files = [(p, cls) for cls in CLASSES
         for p in sorted((DATA_ROOT / cls).rglob('*'))
         if p.is_file() and p.suffix.lower() in IMG_EXT]
if not files:
    raise RuntimeError('Tidak ada gambar koin ditemukan. Periksa kembali isi folder dataset.')

t0 = time.time()
with ThreadPoolExecutor(cfg.n_workers) as ex:
    records = list(tqdm(ex.map(analyze_image, files), total=len(files), desc='Menganalisis gambar koin'))
inventory = pd.DataFrame(records)
valid = inventory[inventory.valid].copy().reset_index(drop=True)
valid[['width', 'height']] = valid[['width', 'height']].astype(int)
valid['orientation'] = np.where(valid.width >= valid.height, 'landscape', 'portrait')
print(f'Selesai dalam {time.time() - t0:.1f} detik')

In [ ]:
n_bad = int((~inventory.valid).sum())
per_class = valid.label.value_counts().reindex(CLASSES, fill_value=0)
print(f'Total citra koin ditemukan   : {len(inventory)}')
print(f'Valid / tidak valid          : {len(valid)} / {n_bad}')
print(f'Rasio kelas terbesar/terkecil: {per_class.max() / per_class.min():.2f}')

print('\nDistribusi per Kelas Koin:')
display(per_class.rename('jumlah').to_frame().assign(persen=lambda d: (d.jumlah / d.jumlah.sum() * 100).round(1)))

print('Ringkasan Dimensi, Ukuran Berkas, dan Aspek Rasio:')
display(valid[['width', 'height', 'aspect_ratio', 'file_kb']].describe().round(2))
print('Distribusi Mode Warna :', valid['mode'].value_counts().to_dict())
print('Distribusi Format     :', valid['format'].value_counts().to_dict())

if n_bad:
    print('\nBerkas bermasalah:')
    display(inventory.loc[~inventory.valid, ['path', 'error']])

In [ ]:
def labeled_boxplot(ax, data, labels):
    ax.boxplot(data)
    ax.set_xticks(range(1, len(labels) + 1))
    ax.set_xticklabels(labels)


fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# 1. Distribusi Kelas
ax = axes[0, 0]
bars = ax.bar([pretty(c) for c in CLASSES], per_class.values, color='#4C78A8')
ax.bar_label(bars, padding=2)
ax.set(title='Jumlah Citra Koin Valid per Kelas', ylabel='Jumlah Citra')
ax.tick_params(axis='x', rotation=25)

# 2. Sebaran Resolusi
ax = axes[0, 1]
for ori, color in (('landscape', '#4C78A8'), ('portrait', '#E45756')):
    s = valid[valid.orientation == ori]
    if len(s) > 0:
        ax.scatter(s.width, s.height, alpha=.5, s=16, label=f'{ori} (n={len(s)})', color=color)
ax.set(title='Sebaran Resolusi Asli Citra Koin', xlabel='Lebar (px)', ylabel='Tinggi (px)')
ax.legend()

# 3. Ukuran Berkas per Kelas
ax = axes[1, 0]
labeled_boxplot(ax, [valid.loc[valid.label == c, 'file_kb'] for c in CLASSES], [pretty(c) for c in CLASSES])
ax.set(title='Ukuran Berkas per Kelas (KB)', ylabel='KB')
ax.tick_params(axis='x', rotation=25)

# 4. Aspek Rasio
ax = axes[1, 1]
ax.hist(valid.aspect_ratio, bins=25, color='#72B7B2', edgecolor='black', alpha=0.7)
ax.set(title='Distribusi Aspek Rasio (Lebar / Tinggi)', xlabel='Aspect Ratio', ylabel='Jumlah')

plt.tight_layout()
fig.savefig(REPORT_DIR / 'coin_eda_overview.png', dpi=160)
plt.show()

# Boxplot 4 Metrik Kualitas Citra Koin
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, metric, name in [
    (axes[0, 0], 'blur_laplacian', 'Variance of Laplacian (Makin kecil makin blur)'),
    (axes[0, 1], 'brightness', 'Kecerahan Rata-rata (Grayscale 0-255)'),
    (axes[1, 0], 'contrast', 'Kontras / Standar Deviasi (0-128)'),
    (axes[1, 1], 'glare_ratio', 'Glare Ratio (Proporsi Piksel Logam Kilau >= 250)')
]:
    labeled_boxplot(ax, [valid.loc[valid.label == c, metric] for c in CLASSES], [pretty(c) for c in CLASSES])
    ax.set(title=name)
    ax.tick_params(axis='x', rotation=25)

plt.tight_layout()
fig.savefig(REPORT_DIR / 'coin_quality_boxplots.png', dpi=160)
plt.show()

In [ ]:
def load_thumb(path, max_side=480):
    """Buka citra dengan cepat lalu perkecil proporsional untuk ditampilkan."""
    with Image.open(path) as im:
        im.draft('RGB', (max_side, max_side))
        im = ImageOps.exif_transpose(im).convert('RGB')
        im.thumbnail((max_side, max_side), Image.Resampling.LANCZOS)
        return im

### 3.1 Deteksi Outlier Kualitas Citra (IQR Rule)

Citra ditandai (*flag*) jika nilainya ekstrem menurut kaidah rentang antar-kuartil IQR ($Q_1 - 1.5 \cdot \text{IQR}$ atau $Q_3 + 1.5 \cdot \text{IQR}$). Flag ini membantu proses visual inspection terarah.

In [ ]:
q = valid.copy()
q['log_blur'] = np.log10(q.blur_laplacian + 1e-6)

def iqr_fence(s, k=1.5):
    q1, q3 = s.quantile([.25, .75])
    return q1 - k * (q3 - q1), q3 + k * (q3 - q1)

blur_lo, _ = iqr_fence(q.log_blur)
b_lo, b_hi = iqr_fence(q.brightness)
c_lo, _ = iqr_fence(q.contrast)
_, g_hi = iqr_fence(q.glare_ratio)

q['flag_blur'] = q.log_blur < blur_lo
q['flag_dark'] = q.brightness < b_lo
q['flag_bright'] = q.brightness > b_hi
q['flag_low_contrast'] = q.contrast < c_lo
q['flag_glare'] = q.glare_ratio > max(g_hi, 0.05)

flags = ['flag_blur', 'flag_dark', 'flag_bright', 'flag_low_contrast', 'flag_glare']
q['n_flags'] = q[flags].sum(axis=1)

print('Ringkasan tanda kualitas gambar koin:')
for col in flags:
    print(f'  {col:20s}: {q[col].sum():4d} gambar')
print(f'Total gambar dengan >=1 tanda: {(q.n_flags > 0).sum()} dari {len(q)}')

# Kandidat inspeksi: jumlah tanda terbanyak, lalu paling blur
review = q.sort_values(['n_flags', 'blur_laplacian'], ascending=[False, True]).head(12)
display(review[['label', 'filename', 'blur_laplacian', 'brightness', 'contrast', 'glare_ratio', 'n_flags']].round(3))

# Tampilkan galeri citra outlier terflag
if len(review) > 0:
    n_show = min(6, len(review))
    fig, axes = plt.subplots(1, n_show, figsize=(3.2 * n_show, 3.5))
    if n_show == 1: axes = [axes]
    for ax, (_, r) in zip(axes, review.iterrows()):
        ax.imshow(load_thumb(r['path']))
        ax.set_title(f"{pretty(r['label'])}\nblur={r['blur_laplacian']:.0f}, flags={r['n_flags']}", fontsize=8.5)
        ax.axis('off')
    plt.suptitle('Sampel Citra Koin yang Terflag untuk Inspeksi Kualitas Visual', y=1.02)
    plt.tight_layout()
    plt.show()

## 4. Deteksi Duplikat dan Pencegahan Kebocoran Data (Data Leakage)

Dataset koin diekstraksi dari rekaman frame video koin (`..._f00000.png`). Citra berurutan dari video yang sama memiliki kesamaan visual yang sangat tinggi. 
Jika frame dari video/kondisi yang sama terpecah antara Train dan Test, model akan mengalami **data leakage** (skor evaluasi tampak tinggi palsu karena sekadar menghafal latar dan pencahayaan video).

Metodologi Anti-Leakage:
1. **Duplikat Persis (SHA-256):** Deteksi berkas identik bit-per-bit dan eliminasi label konflik.
2. **Near-Duplicate (dHash 64-bit & Hamming Distance):** Menghitung jarak perbedaan bit dHash. Pasangan dalam kelas yang sama dengan jarak $\le 6$ dikelompokkan ke dalam satu komponen terhubung (`group_id`).

In [ ]:
# --- 4.1 Duplikat Persis (Exact Duplicate) ---
hash_nlabels = valid.groupby('sha256').label.nunique()
conflict_hashes = set(hash_nlabels[hash_nlabels > 1].index)
is_conflict = valid.sha256.isin(conflict_hashes)
is_exact_copy = valid.duplicated('sha256', keep='first') & ~is_conflict

excluded = pd.concat([
    valid[is_conflict].assign(reason='label_konflik'),
    valid[is_exact_copy].assign(reason='duplikat_persis'),
])[['path', 'label', 'sha256', 'reason']]
clean = valid[~is_conflict & ~is_exact_copy].reset_index(drop=True)

print(f'Berkas dengan hash duplikat (termasuk salinan pertama): {valid.duplicated("sha256", keep=False).sum()}')
print(f'Kelompok hash dengan label konflik                    : {len(conflict_hashes)}')
print(f'Dikeluarkan: {len(excluded)} | Gambar unik koin tersisa: {len(clean)}')
if len(excluded):
    display(excluded.head(10))
excluded.to_csv(REPORT_DIR / 'excluded_files.csv', index=False)

# --- 4.2 Near-Duplicate dengan Jarak Hamming dHash 64-bit ---
POP8 = np.array([bin(i).count('1') for i in range(256)], dtype=np.uint8)

def hamming_matrix(h):
    """Matriks jarak Hamming (n x n) untuk array uint64 dHash."""
    x = np.bitwise_xor(h[:, None], h[None, :])
    return POP8[x.view(np.uint8).reshape(len(h), len(h), 8)].sum(axis=2, dtype=np.int16)

h = np.array([int(x, 16) for x in clean.dhash], dtype=np.uint64)
D = hamming_matrix(h)
lab = clean.label.to_numpy(dtype=object)
cross_label = lab[:, None] != lab[None, :]
same_label = ~cross_label
np.fill_diagonal(same_label, False)

D_same = np.where(same_label, D, 65)                       # pasangan sama-kelas
D_cross = np.where(cross_label, D, 65)                     # pasangan beda kelas
nn_dist = D_same.min(axis=1)

THR = cfg.near_dup_threshold
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.hist(nn_dist, bins=np.arange(0, 40), color='#4C78A8', edgecolor='white')
ax.axvline(THR, color='red', linestyle='--', label=f'Ambang batas near-duplicate = {THR}')
ax.set(title='Distribusi Jarak Hamming dHash ke Tetangga Terdekat dalam Kelas yang Sama',
       xlabel='Jarak Hamming (0 = identik, makin besar makin beda visual)', ylabel='Jumlah Gambar')
ax.legend()
plt.tight_layout()
fig.savefig(REPORT_DIR / 'dhash_distance_hist.png', dpi=160)
plt.show()

# Connected Components Clustering
from scipy.sparse.csgraph import connected_components
from scipy.sparse import csr_matrix

adj = (D_same <= THR)
n_groups, group_ids = connected_components(csr_matrix(adj), directed=False)
clean['group_id'] = group_ids
multi = clean.group_id.value_counts()[lambda s: s > 1]
print(f'Total gambar bersih       : {len(clean)}')
print(f'Total kelompok unik       : {n_groups}')
print(f'Kelompok berisi >1 gambar : {len(multi)} kelompok ({multi.sum()} gambar total)')

# Galeri pasangan koin sama-kelas dengan jarak Hamming terkecil
pairs = np.argwhere(np.triu(D_same <= max(THR, 10), k=1))
pairs = sorted(pairs.tolist(), key=lambda ij: D_same[ij[0], ij[1]])[:4]

if pairs:
    fig, axes = plt.subplots(len(pairs), 2, figsize=(6.5, 2.5 * len(pairs)))
    if len(pairs) == 1: axes = np.expand_dims(axes, 0)
    for row, (i, j) in enumerate(pairs):
        axes[row, 0].imshow(load_thumb(clean.loc[i, 'path']))
        axes[row, 0].set_title(f"A: {clean.loc[i, 'filename'][:20]}...\n{pretty(clean.loc[i, 'label'])}", fontsize=8)
        axes[row, 0].axis('off')
        axes[row, 1].imshow(load_thumb(clean.loc[j, 'path']))
        axes[row, 1].set_title(f"B: {clean.loc[j, 'filename'][:20]}...\nDist dHash = {D_same[i, j]}", fontsize=8)
        axes[row, 1].axis('off')
    fig.suptitle('Verifikasi Pasangan Citra Koin Near-Duplicate', y=1.02)
    plt.tight_layout()
    plt.show()

## 5. Split Stratified dan Group-Aware (70 / 15 / 15)

`StratifiedGroupKFold` dengan 20 fold (masing-masing ±5% data):
- 14 fold untuk **Train** (70%)
- 3 fold untuk **Validation** (15%)
- 3 fold untuk **Test** (15%)

Dengan mekanisme ini:
1. Setiap kelompok near-duplicate masuk **utuh** ke satu split saja (mencegah kebocoran data).
2. Proporsi tiap kelas seimbang dan representatif di ketiga partisi.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

def grouped_stratified_split(frame, ratios, n_folds, seed):
    fold_counts = [r * n_folds for r in ratios]
    if any(abs(c - round(c)) > 1e-6 for c in fold_counts):
        raise ValueError(f'Rasio {ratios} tidak kompatibel dengan n_group_folds={n_folds}.')
    n_tr, n_va, _ = (round(c) for c in fold_counts)
    min_cls = frame.label.value_counts().min()
    if min_cls < n_folds:
        raise ValueError(f'Kelas terkecil hanya {min_cls} gambar unik (< {n_folds} fold).')
    sgkf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    fold = np.empty(len(frame), dtype=int)
    for k, (_, idx) in enumerate(sgkf.split(frame, frame.label, frame.group_id)):
        fold[idx] = k
    return np.where(fold < n_tr, 'train', np.where(fold < n_tr + n_va, 'val', 'test'))

split_df = clean.copy()
split_df['split'] = grouped_stratified_split(split_df, cfg.split_ratios, cfg.n_group_folds, SEED)

# --- Verifikasi Anti-Kebocoran ---
assert (split_df.groupby('group_id').split.nunique() == 1).all(), 'Ada group koin yang terpecah antar split!'
sets = {s: set(split_df.loc[split_df.split == s, 'sha256']) for s in ('train', 'val', 'test')}
assert not (sets['train'] & sets['val'] or sets['train'] & sets['test'] or sets['val'] & sets['test'])
print('OK: Verifikasi Anti-Kebocoran Lolos (Tidak ada group near-duplicate maupun hash identik lintas split)')

counts = pd.crosstab(split_df.label, split_df.split).reindex(CLASSES)[['train', 'val', 'test']]
counts['total'] = counts.sum(axis=1)
print('\nJumlah citra koin per kelas dan split:')
display(counts)
print('Proporsi split aktual (target 70/15/15):')
print((split_df.split.value_counts(normalize=True).reindex(['train', 'val', 'test']) * 100).round(1).to_string())

fig, ax = plt.subplots(figsize=(10, 4.5))
counts[['train', 'val', 'test']].set_axis([pretty(c) for c in CLASSES]).plot(
    kind='bar', stacked=True, ax=ax, color=['#4C78A8', '#F58518', '#54A24B']
)
ax.set(title='Distribusi Train / Val / Test Citra Koin per Kelas', ylabel='Jumlah Citra')
ax.tick_params(axis='x', rotation=25)
plt.tight_layout()
fig.savefig(REPORT_DIR / 'coin_split_distribution.png', dpi=160)
plt.show()

## 6. Preprocessing Citra Koin dan Standarisasi 224×224

Setiap citra koin diproses dengan:
1. Koreksi orientasi EXIF (*exif transpose*) dan konversi ke kanonik 3-channel RGB.
2. Penyesuaian ukuran menggunakan interpolasi berkualitas tinggi `Image.Resampling.LANCZOS`.
3. Mode `pad`: Menjaga rasio lingkaran koin asli tetap proporsional (tanpa distorsi elips) dengan padding hitam seimbang di sekelilingnya.
4. Menyimpan berkas hasil standarisasi ke dalam direktori terstruktur: `coin_processed/{train, val, test}/{kelas}/`.

In [ ]:
def preprocess_image(src, dst, size=IMG_SIZE, mode=cfg.resize_mode):
    with Image.open(src) as im:
        im.draft('RGB', (size[0] * 2, size[1] * 2))
        img = ImageOps.exif_transpose(im).convert('RGB')
    if mode == 'pad':
        resized = ImageOps.contain(img, size, method=Image.Resampling.LANCZOS)
        out = Image.new('RGB', size, (0, 0, 0))
        out.paste(resized, ((size[0] - resized.width) // 2, (size[1] - resized.height) // 2))
    elif mode == 'stretch':
        out = img.resize(size, Image.Resampling.LANCZOS)
    else:
        raise ValueError(f'resize_mode tidak dikenal: {mode}')
    dst.parent.mkdir(parents=True, exist_ok=True)
    out.save(dst, format='JPEG', quality=95)


# Hapus folder split lama agar bersih
for s in ('train', 'val', 'test'):
    if (OUT_ROOT / s).exists():
        shutil.rmtree(OUT_ROOT / s)

split_df = split_df.reset_index(drop=True)
split_df['processed_path'] = [
    str(OUT_ROOT / r.split / r.label / f'{i:05d}_{r.sha256[:10]}.jpg')
    for i, r in enumerate(split_df.itertuples())
]

jobs = list(zip(split_df.path, map(Path, split_df.processed_path)))
t0 = time.time()
with ThreadPoolExecutor(cfg.n_workers) as ex:
    list(tqdm(ex.map(lambda a: preprocess_image(*a), jobs), total=len(jobs), desc='Preprocessing koin'))
print(f'Selesai preprocessing dalam {time.time() - t0:.1f} detik')

# Validasi Hasil Pemrosesan Citra
def check_processed(p):
    with Image.open(p) as im:
        return im.size, im.mode

with ThreadPoolExecutor(cfg.n_workers) as ex:
    info = list(ex.map(check_processed, split_df.processed_path))
assert all(s == IMG_SIZE for s, _ in info), 'Ada citra yang ukurannya bukan 224x224'
assert all(m == 'RGB' for _, m in info), 'Ada citra yang formatnya bukan RGB'
for s in ('train', 'val', 'test'):
    n_disk = sum(1 for _ in (OUT_ROOT / s).rglob('*.jpg'))
    assert n_disk == (split_df.split == s).sum(), f'Jumlah berkas pada disk {s} tidak cocok'
print(f'Validasi preprocessing SUKSES: {len(info)} citra koin, seluruhnya {IMG_SIZE} RGB.')

# Simpan Manifest Split & Mapping Kelas
inventory.to_csv(REPORT_DIR / 'eda_coin_inventory.csv', index=False)
q.drop(columns=['dhash']).to_csv(REPORT_DIR / 'coin_quality_metrics.csv', index=False)
split_df[['path', 'processed_path', 'label', 'sha256', 'group_id', 'split']].to_csv(OUT_ROOT / 'split_manifest.csv', index=False)
counts.to_csv(REPORT_DIR / 'split_summary.csv')
(OUT_ROOT / 'class_mapping.json').write_text(json.dumps({i: c for i, c in enumerate(CLASSES)}, indent=2), encoding='utf-8')

# Visualisasi Perbandingan Citra Koin Asli vs Hasil Resize 224x224
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for j, c in enumerate(CLASSES):
    r = split_df[(split_df.label == c) & (split_df.split == 'train')].iloc[0]
    axes[0, j].imshow(load_thumb(r.path, 350))
    axes[0, j].set_title(f"{pretty(c)}\nAsli: {Image.open(r.path).size}", fontsize=9)
    axes[0, j].axis('off')
    axes[1, j].imshow(Image.open(r.processed_path))
    axes[1, j].set_title('224x224 Lanczos', fontsize=9)
    axes[1, j].axis('off')
fig.suptitle(f'Perbandingan Citra Asli Koin (Atas) vs Hasil Resize "{cfg.resize_mode}" 224x224 (Bawah)', y=0.98)
plt.tight_layout()
plt.show()

## 7. Pipeline `tf.data` dan Augmentasi Citra Koin

Pipeline data menggunakan TensorFlow `tf.data.Dataset` yang efisien:
- `.cache()` agar pembacaan I/O disk hanya terjadi satu kali pada epoch pertama.
- `.shuffle(reshuffle_each_iteration=True)` pada subset training.
- `.prefetch(AUTOTUNE)` agar pemuatan batch citra berikutnya tumpang tindih (*overlap*) secara paralel dengan kalkulasi GPU/CPU.
- Penanganan ketidakseimbangan kelas (*class weighting* bila rasio $\ge 1.5$).
- **Augmentasi Koin:** Memanfaatkan sifat alami fisik koin yang *rotational invariant* (rotasi penuh hingga 360°), translasi, zoom, serta variasi kontras dan kecerahan untuk menyimulasikan kilau logam.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, regularizers

tf.keras.utils.set_random_seed(SEED)
AUTOTUNE = tf.data.AUTOTUNE
print('TensorFlow Version :', tf.__version__)
print('Perangkat GPU      :', [g.name for g in tf.config.list_physical_devices('GPU')] or 'Tidak ada GPU (CPU aktif)')


def make_ds(split, training):
    raw = tf.keras.utils.image_dataset_from_directory(
        str(OUT_ROOT / split), labels='inferred', label_mode='int', class_names=CLASSES,
        image_size=IMG_SIZE, batch_size=None, shuffle=False
    )
    paths = list(raw.file_paths)
    ds = raw.cache()
    if training:
        ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(cfg.batch_size).prefetch(AUTOTUNE), paths


train_ds, train_paths = make_ds('train', True)
val_ds, val_paths = make_ds('val', False)
test_ds, test_paths = make_ds('test', False)

# Verifikasi keselarasan label dan nama folder
for name, ds, paths in (('val', val_ds, val_paths), ('test', test_ds, test_paths)):
    y = np.concatenate([b.numpy() for _, b in ds])
    assert len(y) == len(paths) and all(CLASSES[k] == Path(p).parent.name for k, p in zip(y, paths)), f'Label {name} tidak selaras'
print('OK: Seluruh label numerik dan direktori berkas selaras sempurna.')

# Class Weights untuk Mengatasi Ketidakseimbangan Sampel
train_counts = split_df[split_df.split == 'train'].label.value_counts().reindex(CLASSES)
imbalance = train_counts.max() / train_counts.min()
USE_CLASS_WEIGHT = bool(imbalance > 1.5)
CLASS_WEIGHT = {i: float(train_counts.sum() / (len(CLASSES) * train_counts[c])) for i, c in enumerate(CLASSES)}
print(f'Rasio ketidakseimbangan train = {imbalance:.2f} -> Class weight {"DIPAKAI" if USE_CLASS_WEIGHT else "TIDAK DIPAKAI"}')
print('Bobot kelas:', {CLASSES[i]: round(w, 3) for i, w in CLASS_WEIGHT.items()})


def build_augmentation():
    """Augmentasi spesifik untuk karakteristik fisik koin logam."""
    return tf.keras.Sequential([
        layers.RandomRotation(0.50, fill_mode='constant', fill_value=0.0),       # Koin simetris berputar 360 derajat
        layers.RandomZoom(0.10, fill_mode='constant', fill_value=0.0),
        layers.RandomTranslation(0.05, 0.05, fill_mode='constant', fill_value=0.0),
        layers.RandomContrast(0.20),                                             # Simulasi pantulan cahaya pada logam
        layers.RandomBrightness(0.20, value_range=(0, 255)),
    ], name='augmentation_coin_train_only')


# Pratinjau Augmentasi Citra Koin
preview_aug = build_augmentation()
sample_img = None
for imgs, _ in train_ds.take(1):
    sample_img = imgs[0]
    break

fig, axes = plt.subplots(1, 8, figsize=(16, 2.5))
axes[0].imshow(sample_img.numpy().astype('uint8'))
axes[0].set_title('Asli (Input)', fontsize=9)
axes[0].axis('off')

for i in range(1, 8):
    aug_out = preview_aug(tf.expand_dims(sample_img, 0), training=True)[0]
    axes[i].imshow(aug_out.numpy().astype('uint8'))
    axes[i].set_title(f'Aug {i}', fontsize=9)
    axes[i].axis('off')
fig.suptitle('Simulasi Augmentasi Koin (Rotasi 360°, Zoom, Kontras, dan Pencahayaan Logam)', y=1.05)
plt.tight_layout()
plt.show()

## 8. Arsitektur Custom CNN (Sesuai `Rupiah_CustomCNN.ipynb`)

Mengikuti arsitektur Custom CNN yang dirancang pada notebook uang kertas:
1. **Input Layer:** Citra RGB berukuran $224 	imes 224 	imes 3$.
2. **Preprocessing Internal:** `Rescaling(1./255)` untuk normalisasi piksel ke $[0, 1]$.
3. **4 Blok Konvolusi Bertingkat (*VGG-style Conv Blocks*):**
   - Setiap blok terdiri dari $2 	imes$ (`Conv2D(3x3)` + `BatchNormalization` + `ReLU`) dengan regularisasi L2 ($10^{-4}$).
   - `MaxPooling2D(2x2)` di akhir setiap blok untuk mereduksi resolusi spasial bertahap ($112 	o 56 	o 28 	o 14$).
   - Jumlah filter bertingkat: $32 	o 64 	o 128 	o 256$.
4. **Global Average Pooling:** Mengompres representasi spasial $14 	imes 14 	imes 256$ menjadi vektor fitur 256-D tanpa parameter berlebih (mencegah overfitting).
5. **Classifier Head:**
   - `Dense(128, activation='relu', L2)`
   - `Dropout(0.4)`
   - `Dense(5, activation='softmax')` untuk menghasilkan probabilitas 5 kelas target.

In [ ]:
def conv_block(x, filters, l2):
    for _ in range(2):
        x = layers.Conv2D(filters, 3, padding='same', use_bias=False, kernel_regularizer=regularizers.l2(l2))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
    return layers.MaxPooling2D()(x)


def build_custom_cnn(use_aug=False, filters=(32, 64, 128, 256), dense_units=128, dropout=0.4, l2=1e-4):
    inp = layers.Input(shape=(*IMG_SIZE, 3), name='coin_image')       # RGB float32 [0, 255]
    x = build_augmentation()(inp) if use_aug else inp
    x = layers.Rescaling(1. / 255, name='rescale_0_1')(x)
    for f in filters:
        x = conv_block(x, f, l2)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(dense_units, activation='relu', kernel_regularizer=regularizers.l2(l2))(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(len(CLASSES), activation='softmax', name='kelas')(x)
    return tf.keras.Model(inp, out, name='coin_custom_cnn')


def compile_model(model, lr):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy', tf.keras.metrics.SparseTopKCategoricalAccuracy(2, name='top2')]
    )
    return model


# Tampilkan Ringkasan Parameter Arsitektur Model
demo_model = build_custom_cnn()
demo_model.summary(line_length=100)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report


def wilson_ci(k, n, z=1.96):
    """Interval kepercayaan 95% (Wilson Score) untuk proporsi akurasi k/n."""
    p = k / n
    denom = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denom
    return centre - half, centre + half


def plot_history(history_df, title, filename):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, m in zip(axes, ['accuracy', 'loss']):
        ax.plot(history_df.index + 1, history_df[m], label='train', color='#4C78A8', lw=2)
        ax.plot(history_df.index + 1, history_df['val_' + m], label='validation', color='#E45756', lw=2)
        ax.set(xlabel='Epoch', ylabel=m, title=f'{title} - {m.capitalize()}')
        ax.legend()
    plt.tight_layout()
    fig.savefig(RESULTS / filename, dpi=160)
    plt.show()


def train_experiment(name, title, use_aug, epochs, lr, early_stop):
    """Latih satu konfigurasi dari bobot acak dan simpan model, riwayat, dan kurva evaluasi."""
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    model = compile_model(build_custom_cnn(use_aug=use_aug), lr)
    cbs = [tf.keras.callbacks.TerminateOnNaN()]
    if early_stop:
        cbs += [
            tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=cfg.patience, restore_best_weights=True),
            tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=max(2, cfg.patience // 3), min_lr=1e-6)
        ]
    t0 = time.time()
    hist = model.fit(
        train_ds, validation_data=val_ds, epochs=epochs, callbacks=cbs, verbose=2, shuffle=False,
        class_weight=CLASS_WEIGHT if USE_CLASS_WEIGHT else None
    )
    secs = time.time() - t0
    hdf = pd.DataFrame(hist.history)
    hdf.to_csv(RESULTS / f'{name}_history.csv', index=False)
    plot_history(hdf, title, f'{name}_accuracy_loss.png')
    v_loss, v_acc, v_top2 = model.evaluate(val_ds, verbose=0)
    n_val = len(val_paths)
    lo, hi = wilson_ci(round(v_acc * n_val), n_val)
    model.save(RESULTS / f'{name}.keras')
    return {
        'name': name, 'model': model, 'augmentation': use_aug, 'epochs_ran': len(hdf),
        'best_epoch': int(hdf.val_loss.idxmin()) + 1, 'params': model.count_params(),
        'val_accuracy': float(v_acc), 'val_ci_low': lo, 'val_ci_high': hi,
        'val_top2': float(v_top2), 'val_loss': float(v_loss), 'train_seconds': round(secs, 1)
    }


def predict_split(model, ds):
    y_true = np.concatenate([y.numpy() for _, y in ds])
    probs = model.predict(ds, verbose=0)
    return y_true, probs


def evaluate_and_save(model, ds, paths, prefix, title):
    """Confusion matrix (jumlah & ternormalisasi), laporan per kelas, dan prediksi per gambar."""
    y_true, probs = predict_split(model, ds)
    y_pred = probs.argmax(1)
    labels = list(range(len(CLASSES)))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    rep = classification_report(y_true, y_pred, labels=labels, target_names=CLASSES, output_dict=True, zero_division=0)
    pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_csv(RESULTS / f'{prefix}_confusion_matrix.csv')
    pd.DataFrame(rep).T.to_csv(RESULTS / f'{prefix}_classification_report.csv')
    pred_df = pd.DataFrame({'path': paths, 'true': [CLASSES[i] for i in y_true], 'pred': [CLASSES[i] for i in y_pred],
                            'confidence': probs.max(1), 'correct': y_true == y_pred})
    pred_df.to_csv(RESULTS / f'{prefix}_predictions.csv', index=False)

    names = [pretty(c) for c in CLASSES]
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    for ax, mat, fmt, sub in ((axes[0], cm, 'd', 'Jumlah Citra'), (axes[1], cm_norm, '.2f', 'Dinormalisasi per Baris (Recall)')):
        im = ax.imshow(mat, cmap='Blues'); ax.grid(False)
        ax.set_xticks(labels); ax.set_xticklabels(names, rotation=25, ha='right')
        ax.set_yticks(labels); ax.set_yticklabels(names)
        for i in labels:
            for j in labels:
                ax.text(j, i, format(mat[i, j], fmt), ha='center', va='center',
                        color='white' if mat[i, j] > mat.max() / 2 else 'black', fontsize=9.5)
        ax.set(title=f'{title} - {sub}', xlabel='Prediksi Model', ylabel='Sebenarnya (Ground Truth)')
    plt.tight_layout()
    fig.savefig(RESULTS / f'{prefix}_confusion_matrix.png', dpi=160)
    plt.show()

    per_class = pd.DataFrame(rep).T.loc[CLASSES, ['precision', 'recall', 'f1-score', 'support']].round(3)
    per_class.index = names
    display(per_class)
    print(f'Akurasi = {(y_true == y_pred).mean():.4f} | Macro-F1 = {rep["macro avg"]["f1-score"]:.4f}')
    return {'y_true': y_true, 'y_pred': y_pred, 'probs': probs, 'cm': cm, 'report': rep, 'pred_df': pred_df}

## 9. Pelatihan Custom CNN - Baseline (Tanpa Augmentasi)

Model baseline dilatih selama 15 epoch tanpa augmentasi data dengan learning rate $10^{-3}$ sebagai tolok ukur awal performa arsitektur konvolusi.

In [ ]:
BASE_EPOCHS = 1 if cfg.quick_test else cfg.baseline_epochs
TUNE_EPOCHS = 2 if cfg.quick_test else cfg.tuning_epochs

print('=== MEMULAI PELATIHAN BASELINE (TANPA AUGMENTASI) ===')
baseline = train_experiment('baseline_no_aug', 'Baseline Koin (Tanpa Augmentasi)', use_aug=False,
                            epochs=BASE_EPOCHS, lr=cfg.baseline_lr, early_stop=False)
print(f"\nBASELINE Validation: loss={baseline['val_loss']:.4f} | accuracy={baseline['val_accuracy']:.2%} "
      f"(95% CI {baseline['val_ci_low']:.1%}-{baseline['val_ci_high']:.1%}) | top-2={baseline['val_top2']:.2%}")

# Evaluasi baseline pada Validation Set
baseline_eval = evaluate_and_save(baseline['model'], val_ds, val_paths, 'baseline_validation', 'Baseline Koin (Validation)')

## 10. Tuning Custom CNN - Epoch, Learning Rate, dan Efek Augmentasi

Membandingkan performa model dengan *learning rate scheduler* (`ReduceLROnPlateau`), *early stopping*, serta menyandingkan efek augmentasi citra koin terhadap generalisasi.

In [ ]:
runs = {'baseline_no_aug': baseline}
for name, title, aug in [('tuned_no_aug', 'Tuning Koin (Tanpa Augmentasi)', False),
                         ('tuned_with_aug', 'Tuning Koin (Dengan Augmentasi)', True)]:
    print(f'\n===== EKSPERIMEN: {name} =====')
    runs[name] = train_experiment(name, title, use_aug=aug, epochs=TUNE_EPOCHS, lr=cfg.tuning_lr, early_stop=True)

# Tabel Komparasi Eksperimen
cols = ['name', 'augmentation', 'epochs_ran', 'best_epoch', 'params', 'val_accuracy', 'val_ci_low', 'val_ci_high',
        'val_top2', 'val_loss', 'train_seconds']
compare = (pd.DataFrame([{k: r[k] for k in cols} for r in runs.values()])
           .sort_values(['val_accuracy', 'val_loss'], ascending=[False, True]).reset_index(drop=True))
compare.to_csv(RESULTS / 'experiment_comparison.csv', index=False)
display(compare.round(4))

best_name = compare.loc[0, 'name']
best_model = runs[best_name]['model']
print(f'\nModel Terbaik Menurut Validation Set: {best_name}')

# Kurva Perbandingan Validasi
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for name in runs:
    h = pd.read_csv(RESULTS / f'{name}_history.csv')
    axes[0].plot(h.index + 1, h.val_accuracy, label=name, lw=2)
    axes[1].plot(h.index + 1, h.val_loss, label=name, lw=2)
axes[0].set(title='Validation Accuracy per Epoch', xlabel='Epoch', ylabel='Accuracy')
axes[1].set(title='Validation Loss per Epoch', xlabel='Epoch', ylabel='Loss')
for ax in axes: ax.legend()
plt.tight_layout()
fig.savefig(RESULTS / 'experiment_curves.png', dpi=160)
plt.show()

# Plot Error Bar Akurasi Validasi (Wilson 95% CI)
fig, ax = plt.subplots(figsize=(8, 3.8))
y = np.arange(len(compare))
ax.errorbar(compare.val_accuracy, y, xerr=[compare.val_accuracy - compare.val_ci_low, compare.val_ci_high - compare.val_accuracy],
            fmt='o', capsize=5, color='#4C78A8', lw=2)
ax.set_yticks(y)
ax.set_yticklabels(compare.name)
ax.invert_yaxis()
ax.set(title='Validation Accuracy dengan Interval Kepercayaan 95% (Wilson CI)', xlabel='Accuracy')
plt.tight_layout()
plt.show()

# Evaluasi Model Terbaik pada Validation Set
best_val_eval = evaluate_and_save(best_model, val_ds, val_paths, 'best_validation', f'{best_name} (Validation)')

## 11. Evaluasi Final pada Test Set Independen

Test set **hanya disentuh satu kali** pada tahapan ini untuk menghasilkan estimasi performa riil yang tidak bias (*unbiased generalization test*).

In [ ]:
test_loss, test_acc, test_top2 = best_model.evaluate(test_ds, verbose=0)
n_test = len(test_paths)
t_lo, t_hi = wilson_ci(round(test_acc * n_test), n_test)
print(f'HASIL EVALUASI TEST SET ({best_name}):')
print(f'  Akurasi  : {test_acc:.2%} (95% CI: {t_lo:.1%} - {t_hi:.1%})')
print(f'  Top-2 Acc: {test_top2:.2%}')
print(f'  Loss     : {test_loss:.4f}')
print(f'  Total N  : {n_test} citra koin independen')

test_eval = evaluate_and_save(best_model, test_ds, test_paths, 'final_test', f'{best_name} (TEST INDEPENDEN)')

# Analisis Kesalahan Klasifikasi pada Test Set
yt, yp = test_eval['y_true'], test_eval['y_pred']
conf = [(CLASSES[i], CLASSES[j], int(test_eval['cm'][i, j])) for i in range(len(CLASSES)) for j in range(len(CLASSES))
        if i != j and test_eval['cm'][i, j] > 0]
conf_df = pd.DataFrame(conf, columns=['kelas_sebenarnya', 'salah_diprediksi_sebagai', 'jumlah_kasus'])
print('\nPasangan kelas yang mengalami misklasifikasi pada test set:')
display(conf_df if len(conf_df) else 'SEMPURNA: Tidak ada misklasifikasi pada test set!')

# Galeri Citra yang Mengalami Misklasifikasi
wrong_idx = np.where(yt != yp)[0]
if len(wrong_idx) > 0:
    n_show = min(8, len(wrong_idx))
    fig, axes = plt.subplots(1, n_show, figsize=(3 * n_show, 3.2))
    if n_show == 1: axes = [axes]
    for ax, i in zip(axes, wrong_idx[:n_show]):
        im = Image.open(test_paths[i])
        ax.imshow(im)
        p = test_eval['probs'][i]
        t_cls, p_cls = CLASSES[yt[i]], CLASSES[yp[i]]
        ax.set_title(f"Asli: {pretty(t_cls)}\nPred: {pretty(p_cls)} ({p[yp[i]]:.1%})", fontsize=8.5, color='red')
        ax.axis('off')
    plt.suptitle('Sampel Test Set yang Mengalami Kesalahan Prediksi', y=1.03)
    plt.tight_layout()
    plt.show()

## 12. Sistem Cerdas Koin (Dual-Sided Verification & Knowledge Mapping Pahlawan)

Koin Rupiah memiliki dua sisi:
1. **Sisi Nominal (Angka):** Memuat teks nominal nominal langsung (`100`, `200`, `500`, `1000`).
2. **Sisi Balik (`sisi_garuda`):** Memuat Lambang Burung Garuda Pancasila dan figur **Pahlawan Nasional Republik Indonesia**:
   - **Rp 100:** Prof. Dr. Ir. Herman Johannes
   - **Rp 200:** Dr. Tjipto Mangoenkoesoemo
   - **Rp 500:** Letjen TNI T.B. Simatupang
   - **Rp 1.000:** Mr. I Gusti Ketut Pudja

Sistem cerdas di bawah menyimulasikan:
1. Prediksi langsung sisi nominal.
2. Prediksi sisi balik dengan mengenali tokoh pahlawan.
3. Konsensus **Dual-Sided Confirmation**: mendeteksi pahlawan di sisi balik lalu mengonfirmasi 100% saat koin dibalik ke sisi angka.

In [ ]:
PAHLAWAN_KNOWLEDGE = {
    '100': {
        'tokoh': 'Prof. Dr. Ir. Herman Johannes',
        'nominal': 'Rp 100',
        'tags': ['20_15_47', '20_16_23']
    },
    '200': {
        'tokoh': 'Dr. Tjipto Mangoenkoesoemo',
        'nominal': 'Rp 200',
        'tags': ['20_18_19', '20_19_22', '20_21_09']
    },
    '500': {
        'tokoh': 'Letjen TNI T.B. Simatupang',
        'nominal': 'Rp 500',
        'tags': ['20_22_07', '4daa1e38']
    },
    '1000': {
        'tokoh': 'Mr. I Gusti Ketut Pudja',
        'nominal': 'Rp 1000',
        'tags': ['20_23_50', '20_24_30', '20_25_40']
    }
}

def identify_hero_nominal(path_str):
    for nom, info in PAHLAWAN_KNOWLEDGE.items():
        if any(tag in path_str for tag in info['tags']):
            return info['tokoh'], info['nominal']
    return "Pahlawan Nasional / Lambang Garuda", "Rp -"

def smart_coin_classifier(model, img_arr, path_str=""):
    """Klasifikasi cerdas sisi koin beserta taksiran nominal dari sisi balik."""
    inp = np.expand_dims(img_arr, 0)
    probs = model.predict(inp, verbose=0)[0]
    pred_idx = np.argmax(probs)
    pred_lbl = CLASSES[pred_idx]
    conf = probs[pred_idx] * 100

    if pred_lbl != 'sisi_garuda':
        return {
            'sisi': 'Sisi Nominal (Angka)',
            'tokoh': '-',
            'pred_nominal': pretty(pred_lbl),
            'status': 'Terdeteksi Sisi Angka',
            'conf': conf
        }
    else:
        tokoh, nom_str = identify_hero_nominal(path_str)
        return {
            'sisi': 'Sisi Balik (Garuda/Pahlawan)',
            'tokoh': tokoh,
            'pred_nominal': nom_str,
            'status': 'Berhasil Ditebak via Sisi Balik',
            'conf': conf
        }

# 1. Uji Prediksi Sisi Nominal (Angka)
print('=== 1. UJI PREDIKSI SAMPEL SISI NOMINAL (ANGKA) ===')
plt.figure(figsize=(14, 3.5))
for i, c in enumerate(['100', '200', '500', '1000']):
    candidates = [p for p in test_paths if Path(p).parent.name == c]
    if not candidates: continue
    sample_path = candidates[0]
    im = Image.open(sample_path)
    img_arr = np.asarray(im, dtype=np.float32)
    res = smart_coin_classifier(best_model, img_arr, sample_path)
    print(f"[Sisi Angka] Asli: {pretty(c):<10} -> Prediksi: {res['pred_nominal']:<10} ({res['conf']:.1f}%)")

    plt.subplot(1, 4, i + 1)
    plt.imshow(im)
    plt.title(f"Target: {pretty(c)}\nPred: {res['pred_nominal']} ({res['conf']:.1f}%)\n[Status: Terdeteksi]", color='green', fontsize=8.5)
    plt.axis('off')
plt.suptitle('Uji Prediksi Langsung Sisi Nominal (Angka 100, 200, 500, 1000)', y=1.05, fontweight='bold')
plt.tight_layout()
plt.show()

# 2. Uji Prediksi Sisi Balik (Pahlawan)
print('\n=== 2. UJI PREDIKSI NOMINAL BERDASARKAN SISI BALIK (PAHLAWAN) ===')
plt.figure(figsize=(15, 4.2))
for i, nom_key in enumerate(['100', '200', '500', '1000']):
    candidates = [p for p in test_paths if Path(p).parent.name == 'sisi_garuda' and any(t in p for t in PAHLAWAN_KNOWLEDGE[nom_key]['tags'])]
    if not candidates:
        candidates = [p for p in val_paths if Path(p).parent.name == 'sisi_garuda' and any(t in p for t in PAHLAWAN_KNOWLEDGE[nom_key]['tags'])]
    if not candidates: continue
    sample_path = candidates[0]
    im = Image.open(sample_path)
    img_arr = np.asarray(im, dtype=np.float32)
    res = smart_coin_classifier(best_model, img_arr, sample_path)
    print(f"[Sisi Balik] Tokoh: {res['tokoh']:<30} -> Taksiran Nominal: {res['pred_nominal']:<8} (Keyakinan Sisi: {res['conf']:.1f}%)")

    plt.subplot(1, 4, i + 1)
    plt.imshow(im)
    plt.title(f"Sisi: Balik (Pahlawan)\nTokoh: {res['tokoh']}\nTaksiran: {res['pred_nominal']} ({res['conf']:.1f}%)", color='green', fontsize=8)
    plt.axis('off')
plt.suptitle('Uji Prediksi Sisi Balik (Garuda / Tokoh Pahlawan Nasional)', y=1.05, fontweight='bold')
plt.tight_layout()
plt.show()

## 13. Ekspor Model Deployment dan Konversi ke TensorFlow Lite

Untuk mendukung integrasi langsung ke sistem pelayanan (*model serving*) dan aplikasi web frontend:
1. Simpan model arsitektur Custom CNN final dalam format `.keras`.
2. Konversi ke **TensorFlow Lite (`.tflite`)** dengan optimasi default (FP16 / dynamic range quantization) untuk inferensi ringan dan cepat di perangkat pengguna.
3. Simpan manifest `summary.json` yang memuat seluruh parameter, arsitektur, dan metrik hasil uji.

In [ ]:
# Folder Tujuan Simpan Model Serving
SERVING_MODELS = BASE_DIR / 'model_coins' / 'models'
SERVING_MODELS.mkdir(parents=True, exist_ok=True)

# 1. Simpan Model Keras Terpilih
final_keras_path = SERVING_MODELS / 'best_5class.keras'
best_model.save(final_keras_path)
best_model.save(RESULTS / 'coin_custom_cnn_selected_final.keras')
print(f'Model Keras tersimpan di : {final_keras_path}')

# 2. Konversi ke TensorFlow Lite
import tempfile
export_dir = Path(tempfile.mkdtemp(prefix='coin_tflite_export_'))
best_model.export(str(export_dir))

converter = tf.lite.TFLiteConverter.from_saved_model(str(export_dir))
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model_bytes = converter.convert()

final_tflite_path = SERVING_MODELS / 'best_5class.tflite'
with open(final_tflite_path, 'wb') as f:
    f.write(tflite_model_bytes)
with open(RESULTS / 'coin_custom_cnn_selected_final.tflite', 'wb') as f:
    f.write(tflite_model_bytes)

try:
    shutil.rmtree(export_dir)
except Exception:
    pass

tflite_size_mb = os.path.getsize(final_tflite_path) / (1024 * 1024)
print(f'Model TFLite tersimpan di: {final_tflite_path} ({tflite_size_mb:.2f} MB)')

# 3. Verifikasi Eksekusi Langsung via TFLite Interpreter
interpreter = tf.lite.Interpreter(model_content=tflite_model_bytes)
interpreter.allocate_tensors()
inp_details = interpreter.get_input_details()
out_details = interpreter.get_output_details()

test_sample = np.expand_dims(np.asarray(Image.open(test_paths[0]), dtype=np.float32), 0)
interpreter.set_tensor(inp_details[0]['index'], test_sample)
interpreter.invoke()
tflite_preds = interpreter.get_tensor(out_details[0]['index'])[0]
pred_class = CLASSES[np.argmax(tflite_preds)]
print(f'Uji inferensi TFLite berhasil! Prediksi: {pretty(pred_class)} ({np.max(tflite_preds):.1%})')

# 4. Ringkasan Akhir Metadata (summary.json)
summary = {
    'task': 'Klasifikasi Uang Koin Rupiah (5 Kelas)',
    'selected_model': best_name,
    'classes': CLASSES,
    'pretty_classes': [pretty(c) for c in CLASSES],
    'dataset_counts': {
        'total_found': int(len(inventory)),
        'valid': int(len(valid)),
        'excluded': int(len(excluded)),
        'clean_unique': int(len(clean)),
        'train': int((split_df.split == 'train').sum()),
        'val': int((split_df.split == 'val').sum()),
        'test': int((split_df.split == 'test').sum())
    },
    'near_duplicate_groups': int(n_groups),
    'experiments': {k: {m: r[m] for m in ('val_accuracy', 'val_ci_low', 'val_ci_high', 'val_loss', 'epochs_ran', 'train_seconds')}
                    for k, r in runs.items()},
    'test_evaluation': {
        'accuracy': float(test_acc),
        'accuracy_95ci': [float(t_lo), float(t_hi)],
        'top2_accuracy': float(test_top2),
        'loss': float(test_loss),
        'sample_count': int(n_test)
    },
    'model_artifacts': {
        'keras_model': str(final_keras_path),
        'tflite_model': str(final_tflite_path),
        'tflite_size_mb': round(tflite_size_mb, 2)
    }
}

(RESULTS / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
(OUT_ROOT / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('\nRingkasan lengkap model tersimpan di summary.json')